: 

In [2]:

# FIXED-BATCH + JITTERED-NOISE STOCHASTIC SQP-PINN (JAX)
# FOR REDUCED COUPLED NEUTRON / FUEL-HEAT / COOLANT-HEAT
#
# Network outputs: [phi, Ts, Tf]
#
#   phi : neutron flux in fuel
#   Ts  : fuel temperature in fuel
#   Tf  : coolant temperature in coolant
#
# No Navier-Stokes. No pressure.
# Coolant advection is prescribed by constant upward velocity vy.
#
# IMPORTANT:
#   - base collocation/boundary/interface/IC point sets are sampled ONCE
#   - each iteration uses fresh jittered versions of those fixed sets
#   - no resampling during training
#
# Objective:
#   PDE residuals on large fixed base sets (with fresh jitter each iter)
#
# Hard constraints:
#   PDE anchors on smaller fixed base sets (with fresh jitter each iter)
#   BC / IC / interface constraints
#
# This follows your Burgers / test style:
#   - train_sqp(...) has the knobs in main()
#   - jitter/noise magnitudes chosen in main()
#   - weights chosen in main()
#   - gamma, L_lip, Gamma_lip etc. chosen in main()
# ============================================================

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_command_buffer="

import time
import math
from functools import partial
import numpy as np

import jax
import jax.numpy as jnp
from jax import random, vmap, jacrev, hessian

# ============================================================
# Precision / dtype
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# GEOMETRY
# ============================================================
Ls = DTYPE(0.4)
Lf = DTYPE(0.6)
Ly = DTYPE(1.0)
T_end = DTYPE(1.0)

x_min = DTYPE(0.0)
x_max = DTYPE(Ls + Lf)
y_min = DTYPE(0.0)
y_max = DTYPE(Ly)
t_min = DTYPE(0.0)
t_max = DTYPE(T_end)

# ============================================================
# SAMPLE COUNTS (edit here if you want different set sizes)
# ============================================================
# Objective PDE sets (large, soft)
NX_OBJ_PHI, NY_OBJ_PHI, NT_OBJ_PHI = 15, 15, 10
NX_OBJ_TS,  NY_OBJ_TS,  NT_OBJ_TS  = 15, 15, 10
NX_OBJ_TF,  NY_OBJ_TF,  NT_OBJ_TF  = 15, 15, 10

# PDE anchor sets (smaller, hard)
NX_CON_PHI, NY_CON_PHI, NT_CON_PHI = 6 ,6, 6
NX_CON_TS,  NY_CON_TS,  NT_CON_TS  = 6, 6, 6
NX_CON_TF,  NY_CON_TF,  NT_CON_TF  = 6, 6, 6

K_CON_PHI = NX_CON_PHI * NY_CON_PHI * NT_CON_PHI
K_CON_TS  = NX_CON_TS  * NY_CON_TS  * NT_CON_TS
K_CON_TF  = NX_CON_TF  * NY_CON_TF  * NT_CON_TF

N_PER_CELL = 1
N_BC_PER_BIN = 1
N_IC_PER_BIN = 1
N_IF_PER_BIN = 1

# BC counts
K_BC_PHI_LEFT = 60
K_BC_PHI_Y0   = 60
K_BC_PHI_Y1   = 60
K_BC_PHI_IF   = 60

K_BC_TS_X0    = 60
K_BC_TS_Y0    = 60
K_BC_TS_Y1    = 60

K_BC_TF_Y0    = 60  # coolant bottom Dirichlet Tf=0
K_BC_TF_Y1    = 60   # coolant top zero diffusive flux
K_BC_TF_X1    = 60   # coolant right zero diffusive flux

# IC counts: separate point sets for phi, Ts, Tf
NX_IC, NY_IC = 10, 20
K_IC_PHI = NX_IC * NY_IC
K_IC_TS  = NX_IC * NY_IC
K_IC_TF  = NX_IC * NY_IC

# Interface points
NY_IF, NT_IF = 10, 10
K_IF = NY_IF * NT_IF

# ============================================================
# BLOCK SIZES
# ============================================================
N_PDE_PHI = K_CON_PHI
N_PDE_TS  = K_CON_TS
N_PDE_TF  = K_CON_TF

N_BC_PHI = K_BC_PHI_LEFT + K_BC_PHI_Y0 + K_BC_PHI_Y1 + K_BC_PHI_IF
N_BC_TS  = K_BC_TS_X0 + K_BC_TS_Y0 + K_BC_TS_Y1
N_BC_TF  = K_BC_TF_Y0 + K_BC_TF_Y1 + K_BC_TF_X1
N_IF     = 2 * K_IF
N_IC     = K_IC_PHI + K_IC_TS + K_IC_TF

M_CON = N_PDE_PHI + N_PDE_TS + N_PDE_TF + N_BC_PHI + N_BC_TS + N_BC_TF + N_IF + N_IC

# ============================================================
# TARGETS / BCs / ICs
# ============================================================
@jax.jit
def phi_left_source(y, t):
    return (DTYPE(1.0) - jnp.exp(-DTYPE(5.0) * t)) * (
        DTYPE(1.0) + DTYPE(0.2) * jnp.sin(jnp.pi * y) * jnp.sin(jnp.pi * y)
    )

@jax.jit
def phi_ic_target(y):
    return jnp.zeros_like(y)

@jax.jit
def Ts_ic_target(x, y):
    return jnp.zeros_like(x)

@jax.jit
def Tf_ic_target(x, y):
    return jnp.zeros_like(x)

# ============================================================
# NETWORK
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params

def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]

    x_n = DTYPE(2.0) * (x - x_min) / (x_max - x_min + EPS) - DTYPE(1.0)
    y_n = DTYPE(2.0) * (y - y_min) / (y_max - y_min + EPS) - DTYPE(1.0)
    t_n = DTYPE(2.0) * (t - t_min) / (t_max - t_min + EPS) - DTYPE(1.0)
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)
    # outputs = [phi, Ts, Tf]
    return h

def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    theta = jnp.concatenate(flat_parts).astype(DTYPE)
    return theta, tuple(shapes)

def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)
        W = theta[idx: idx + W_size].reshape(W_shape)
        idx += W_size
        b = theta[idx: idx + b_size].reshape(b_shape)
        idx += b_size
        params.append({"W": W, "b": b})
    return params

# ============================================================
# UTILITIES
# ============================================================
def segment_sum(values, segment_ids, num_segments):
    out = jnp.zeros((num_segments,), dtype=values.dtype)
    return out.at[segment_ids].add(values)

# ============================================================
# STRATIFIED SAMPLING
# ============================================================
def sample_stratified_3d(key, x0, x1, y0, y1, t0, t1, NX, NY, NT):
    dx = (DTYPE(x1) - DTYPE(x0)) / DTYPE(NX)
    dy = (DTYPE(y1) - DTYPE(y0)) / DTYPE(NY)
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(NT)

    it, iy, ix = jnp.meshgrid(jnp.arange(NT), jnp.arange(NY), jnp.arange(NX), indexing="ij")
    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids = (it * (NY * NX) + iy * NX + ix).astype(jnp.int32)

    xb = DTYPE(x0) + DTYPE(ix) * dx
    yb = DTYPE(y0) + DTYPE(iy) * dy
    tb = DTYPE(t0) + DTYPE(it) * dt

    K = NX * NY * NT
    u = random.uniform(key, (K, 3), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = xb + u[:, 0] * dx
    ys = yb + u[:, 1] * dy
    ts = tb + u[:, 2] * dt
    X = jnp.stack([xs, ys, ts], axis=1)
    return X, ids

def sample_bc_time_binned(key, K, *, x_fixed=None, y_fixed=None, t0=t_min, t1=t_max,
                          x_lo=None, x_hi=None, y_lo=None, y_hi=None):
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(K)
    j = jnp.arange(K, dtype=jnp.int32)
    tb = DTYPE(t0) + DTYPE(j) * dt

    key, kt, kr = random.split(key, 3)
    u_t = random.uniform(kt, (K, N_BC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (tb[:, None] + u_t * dt).reshape(-1, 1)
    ids = jnp.repeat(j, N_BC_PER_BIN)

    if x_fixed is not None:
        lo = DTYPE(y_min if y_lo is None else y_lo)
        hi = DTYPE(y_max if y_hi is None else y_hi)
        y = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        x = DTYPE(x_fixed) * jnp.ones_like(y)
        return jnp.concatenate([x, y, ts], axis=1), ids

    if y_fixed is not None:
        lo = DTYPE(x_min if x_lo is None else x_lo)
        hi = DTYPE(x_max if x_hi is None else x_hi)
        x = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        y = DTYPE(y_fixed) * jnp.ones_like(x)
        return jnp.concatenate([x, y, ts], axis=1), ids

    raise ValueError("Provide x_fixed or y_fixed")

def sample_ic_xy(key, x0, x1, y0, y1, NX, NY):
    dx = (DTYPE(x1) - DTYPE(x0)) / DTYPE(NX)
    dy = (DTYPE(y1) - DTYPE(y0)) / DTYPE(NY)

    iy, ix = jnp.meshgrid(jnp.arange(NY), jnp.arange(NX), indexing="ij")
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids0 = (iy * NX + ix).astype(jnp.int32)

    xb = DTYPE(x0) + DTYPE(ix) * dx
    yb = DTYPE(y0) + DTYPE(iy) * dy

    u = random.uniform(key, (NX * NY, N_IC_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (xb[:, None] + u[:, :, 0] * dx).reshape(-1, 1)
    ys = (yb[:, None] + u[:, :, 1] * dy).reshape(-1, 1)
    ts = t_min * jnp.ones_like(xs)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IC_PER_BIN)
    return X, ids

def sample_interface_yt(key):
    dy = (y_max - y_min) / DTYPE(NY_IF)
    dt = (t_max - t_min) / DTYPE(NT_IF)

    it, iy = jnp.meshgrid(jnp.arange(NT_IF), jnp.arange(NY_IF), indexing="ij")
    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)
    ids0 = (it * NY_IF + iy).astype(jnp.int32)

    yb = y_min + DTYPE(iy) * dy
    tb = t_min + DTYPE(it) * dt

    u = random.uniform(key, (NY_IF * NT_IF, N_IF_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    ys = (yb[:, None] + u[:, :, 0] * dy).reshape(-1, 1)
    ts = (tb[:, None] + u[:, :, 1] * dt).reshape(-1, 1)
    xs = DTYPE(Ls) * jnp.ones_like(ys)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IF_PER_BIN)
    return X, ids

# ============================================================
# JITTER + PROJECTIONS
# ============================================================
def jitter_xyt(key, X, sx, sy, st, clip_k=2.0):
    if X.shape[0] == 0 or ((sx <= 0) and (sy <= 0) and (st <= 0)):
        return X

    key, kx, ky, kt = random.split(key, 4)
    dx = DTYPE(sx) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
    dy = DTYPE(sy) * random.normal(ky, (X.shape[0],), dtype=DTYPE)
    dt = DTYPE(st) * random.normal(kt, (X.shape[0],), dtype=DTYPE)

    k = DTYPE(clip_k)
    dx = jnp.clip(dx, -k * DTYPE(sx), k * DTYPE(sx))
    dy = jnp.clip(dy, -k * DTYPE(sy), k * DTYPE(sy))
    dt = jnp.clip(dt, -k * DTYPE(st), k * DTYPE(st))

    x = jnp.clip(X[:, 0] + dx, x_min, x_max)
    y = jnp.clip(X[:, 1] + dy, y_min, y_max)
    t = jnp.clip(X[:, 2] + dt, t_min, t_max)
    return jnp.stack([x, y, t], axis=1)

def project_x_fixed(X, x_fixed):
    if X.shape[0] == 0:
        return X
    return jnp.stack([
        DTYPE(x_fixed) * jnp.ones((X.shape[0],), dtype=DTYPE),
        X[:, 1],
        X[:, 2],
    ], axis=1)

def project_y_fixed(X, y_fixed, *, x_lo=None, x_hi=None):
    if X.shape[0] == 0:
        return X
    x = X[:, 0]
    if (x_lo is not None) or (x_hi is not None):
        lo = DTYPE(x_lo if x_lo is not None else x_min)
        hi = DTYPE(x_hi if x_hi is not None else x_max)
        x = jnp.clip(x, lo, hi)
    y = DTYPE(y_fixed) * jnp.ones((X.shape[0],), dtype=DTYPE)
    return jnp.stack([x, y, X[:, 2]], axis=1)

def project_t0(X):
    if X.shape[0] == 0:
        return X
    return jnp.stack([
        X[:, 0],
        X[:, 1],
        t_min * jnp.ones((X.shape[0],), dtype=DTYPE),
    ], axis=1)

def project_interface(X):
    return project_x_fixed(X, Ls)

# ============================================================
# PDE RESIDUALS
# ============================================================
def forward3(params, xyt):
    return mlp_apply(params, xyt[None, :])[0, :]

def eval_fields_and_derivs(params, X):
    out = vmap(lambda z: forward3(params, z))(X)
    J   = vmap(jacrev(lambda z: forward3(params, z)))(X)
    H   = vmap(hessian(lambda z: forward3(params, z)))(X)
    return out, J, H

def residual_phi(params, X, D_phi, Sigma_a):
    out, J, H = eval_fields_and_derivs(params, X)
    phi = out[:, 0]
    phi_t = J[:, 0, 2]
    phi_xx = H[:, 0, 0, 0]
    phi_yy = H[:, 0, 1, 1]
    lap_phi = phi_xx + phi_yy
    return phi_t - DTYPE(D_phi) * lap_phi + DTYPE(Sigma_a) * phi

def residual_Ts(params, X, rho_s_cp_s, k_s, gamma):
    out, J, H = eval_fields_and_derivs(params, X)
    phi = out[:, 0]
    Ts  = out[:, 1]
    Ts_t  = J[:, 1, 2]
    Ts_xx = H[:, 1, 0, 0]
    Ts_yy = H[:, 1, 1, 1]
    lap_Ts = Ts_xx + Ts_yy
    return DTYPE(rho_s_cp_s) * Ts_t - DTYPE(k_s) * lap_Ts - DTYPE(gamma) * phi

def residual_Tf(params, X, rho_f_cp_f, k_f, vy):
    out, J, H = eval_fields_and_derivs(params, X)
    Tf = out[:, 2]
    Tf_t = J[:, 2, 2]
    Tf_y = J[:, 2, 1]
    Tf_xx = H[:, 2, 0, 0]
    Tf_yy = H[:, 2, 1, 1]
    lap_Tf = Tf_xx + Tf_yy
    return DTYPE(rho_f_cp_f) * Tf_t + DTYPE(vy) * Tf_y - DTYPE(k_f) * lap_Tf

# ============================================================
# OBJECTIVE
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def F_and_g_obj(theta, shapes, X_obj_phi, X_obj_Ts, X_obj_Tf,
                w_obj_phi, w_obj_Ts, w_obj_Tf,
                D_phi, Sigma_a, rho_s_cp_s, k_s, gamma, rho_f_cp_f, k_f, vy):
    def obj_theta(th):
        params = unflatten_params(th, shapes)
        r_phi = residual_phi(params, X_obj_phi, D_phi, Sigma_a)
        r_Ts  = residual_Ts(params, X_obj_Ts, rho_s_cp_s, k_s, gamma)
        r_Tf  = residual_Tf(params, X_obj_Tf, rho_f_cp_f, k_f, vy)
        return (
            DTYPE(w_obj_phi) * jnp.mean(r_phi ** 2)
            + DTYPE(w_obj_Ts) * jnp.mean(r_Ts ** 2)
            + DTYPE(w_obj_Tf) * jnp.mean(r_Tf ** 2)
        )

    val, g = jax.value_and_grad(obj_theta)(theta)
    return val, g

# ============================================================
# CONSTRAINT VECTOR
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector(theta, shapes,
                      X_con_phi, ids_con_phi,
                      X_con_Ts, ids_con_Ts,
                      X_con_Tf, ids_con_Tf,
                      X_phi_left, ids_phi_left,
                      X_phi_y0, ids_phi_y0,
                      X_phi_y1, ids_phi_y1,
                      X_phi_if, ids_phi_if,
                      X_Ts_x0, ids_Ts_x0,
                      X_Ts_y0, ids_Ts_y0,
                      X_Ts_y1, ids_Ts_y1,
                      X_Tf_y0, ids_Tf_y0,
                      X_Tf_y1, ids_Tf_y1,
                      X_Tf_x1, ids_Tf_x1,
                      X_if, ids_if,
                      X_ic_phi, ids_ic_phi,
                      X_ic_Ts, ids_ic_Ts,
                      X_ic_Tf, ids_ic_Tf,
                      D_phi, Sigma_a, rho_s_cp_s, k_s, gamma, rho_f_cp_f, k_f, vy):
    params = unflatten_params(theta, shapes)

    # PDE anchors
    r_phi = residual_phi(params, X_con_phi, D_phi, Sigma_a)
    r_Ts  = residual_Ts(params, X_con_Ts, rho_s_cp_s, k_s, gamma)
    r_Tf  = residual_Tf(params, X_con_Tf, rho_f_cp_f, k_f, vy)

    c_pde_phi = segment_sum(r_phi, ids_con_phi, K_CON_PHI) / DTYPE(N_PER_CELL)
    c_pde_Ts  = segment_sum(r_Ts,  ids_con_Ts,  K_CON_TS)  / DTYPE(N_PER_CELL)
    c_pde_Tf  = segment_sum(r_Tf,  ids_con_Tf,  K_CON_TF)  / DTYPE(N_PER_CELL)

    # scalar helpers
    def phi_fun(z):
        return forward3(params, z)[0]
    def Ts_fun(z):
        return forward3(params, z)[1]
    def Tf_fun(z):
        return forward3(params, z)[2]

    # Neutron BCs
    phiL = mlp_apply(params, X_phi_left)[:, 0]
    yL, tL = X_phi_left[:, 1], X_phi_left[:, 2]
    c_phi_left = segment_sum(phiL - phi_left_source(yL, tL), ids_phi_left, K_BC_PHI_LEFT) / DTYPE(N_BC_PER_BIN)

    phi_y0 = vmap(lambda z: jacrev(phi_fun)(z)[1])(X_phi_y0)
    c_phi_y0 = segment_sum(phi_y0, ids_phi_y0, K_BC_PHI_Y0) / DTYPE(N_BC_PER_BIN)

    phi_y1 = vmap(lambda z: jacrev(phi_fun)(z)[1])(X_phi_y1)
    c_phi_y1 = segment_sum(phi_y1, ids_phi_y1, K_BC_PHI_Y1) / DTYPE(N_BC_PER_BIN)

    phi_x_if = vmap(lambda z: jacrev(phi_fun)(z)[0])(X_phi_if)
    c_phi_if = segment_sum(phi_x_if, ids_phi_if, K_BC_PHI_IF) / DTYPE(N_BC_PER_BIN)

    c_bc_phi = jnp.concatenate([c_phi_left, c_phi_y0, c_phi_y1, c_phi_if], axis=0)

    # Fuel thermal BCs
    Ts_x0 = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_Ts_x0)
    c_Ts_x0 = segment_sum(Ts_x0, ids_Ts_x0, K_BC_TS_X0) / DTYPE(N_BC_PER_BIN)

    Ts_y0 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_Ts_y0)
    c_Ts_y0 = segment_sum(Ts_y0, ids_Ts_y0, K_BC_TS_Y0) / DTYPE(N_BC_PER_BIN)

    Ts_y1 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_Ts_y1)
    c_Ts_y1 = segment_sum(Ts_y1, ids_Ts_y1, K_BC_TS_Y1) / DTYPE(N_BC_PER_BIN)

    c_bc_Ts = jnp.concatenate([c_Ts_x0, c_Ts_y0, c_Ts_y1], axis=0)

    # Coolant thermal BCs
    Tf_bottom = mlp_apply(params, X_Tf_y0)[:, 2]
    c_Tf_y0 = segment_sum(Tf_bottom, ids_Tf_y0, K_BC_TF_Y0) / DTYPE(N_BC_PER_BIN)

    Tf_y1 = vmap(lambda z: jacrev(Tf_fun)(z)[1])(X_Tf_y1)
    c_Tf_y1 = segment_sum(Tf_y1, ids_Tf_y1, K_BC_TF_Y1) / DTYPE(N_BC_PER_BIN)

    Tf_x1 = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_Tf_x1)
    c_Tf_x1 = segment_sum(Tf_x1, ids_Tf_x1, K_BC_TF_X1) / DTYPE(N_BC_PER_BIN)

    c_bc_Tf = jnp.concatenate([c_Tf_y0, c_Tf_y1, c_Tf_x1], axis=0)

    # Interface constraints
    out_if = mlp_apply(params, X_if)
    Ts_if = out_if[:, 1]
    Tf_if = out_if[:, 2]

    c_if_T = segment_sum(Ts_if - Tf_if, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    Tsx_if = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_if)
    Tfx_if = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_if)

    flux_jump = -DTYPE(k_s) * Tsx_if + DTYPE(k_f) * Tfx_if
    c_if_q = segment_sum(flux_jump, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    c_if = jnp.concatenate([c_if_T, c_if_q], axis=0)

    # ICs: separate sets, no masks
    out_phi0 = mlp_apply(params, X_ic_phi)
    out_Ts0  = mlp_apply(params, X_ic_Ts)
    out_Tf0  = mlp_apply(params, X_ic_Tf)

    phi0 = out_phi0[:, 0]
    Ts0  = out_Ts0[:, 1]
    Tf0  = out_Tf0[:, 2]

    y_phi0 = X_ic_phi[:, 1]
    c_ic_phi = segment_sum(phi0 - phi_ic_target(y_phi0), ids_ic_phi, K_IC_PHI) / DTYPE(N_IC_PER_BIN)
    c_ic_Ts  = segment_sum(Ts0 - Ts_ic_target(X_ic_Ts[:, 0], X_ic_Ts[:, 1]), ids_ic_Ts, K_IC_TS) / DTYPE(N_IC_PER_BIN)
    c_ic_Tf  = segment_sum(Tf0 - Tf_ic_target(X_ic_Tf[:, 0], X_ic_Tf[:, 1]), ids_ic_Tf, K_IC_TF) / DTYPE(N_IC_PER_BIN)

    c_ic = jnp.concatenate([c_ic_phi, c_ic_Ts, c_ic_Tf], axis=0)

    return jnp.concatenate([
        c_pde_phi,
        c_pde_Ts,
        c_pde_Tf,
        c_bc_phi,
        c_bc_Ts,
        c_bc_Tf,
        c_if,
        c_ic
    ], axis=0)

@partial(jax.jit, static_argnames=("shapes",))
def C_and_J(theta, shapes, *args):
    def c_fun(th):
        return constraint_vector(th, shapes, *args)

    def c_fun_aux(th):
        c = c_fun(th)
        return c, c

    J, c = jax.jacrev(c_fun_aux, has_aux=True)(theta)
    return c, J

@partial(jax.jit, static_argnames=("shapes",))
def C_only(theta, shapes, *args):
    return constraint_vector(theta, shapes, *args)

# ============================================================
# KKT solve
# ============================================================
@jax.jit
def kkt_solve_once(H, Jac, Grad, Cons, mu_damp):
    n = H.shape[0]
    m = Cons.shape[0]
    I_m = jnp.eye(m, dtype=H.dtype)
    top = jnp.concatenate([H, Jac.T], axis=1)
    bottom = jnp.concatenate([Jac, -mu_damp * I_m], axis=1)
    KKT = jnp.concatenate([top, bottom], axis=0)
    rhs = -jnp.concatenate([Grad, Cons])
    sol = jnp.linalg.solve(KKT, rhs)
    d = sol[:n]
    y = sol[n:]
    return d, y

# ============================================================
# STEP HELPERS
# ============================================================
def cal_tau_mu(H, d, sigma, tau_pre, eps_tau, g, c, mu_eff, y):
    denom = float(g @ d + 0.5 * (d @ (H @ d)))
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    if (denom <= 1e-12) or (delta_c <= 0.0):
        tau_trial = float("inf")
    else:
        tau_trial = (1.0 - sigma) * delta_c / denom
    if tau_pre <= tau_trial:
        return tau_pre
    return min(tau_trial, (1.0 - eps_tau) * tau_pre)

def cal_ksi_mu(d, tau, ksi_old, eps_ksi, g, c, mu_eff, y):
    d2 = float(jnp.linalg.norm(d) ** 2)
    if d2 <= 1e-18 or tau <= 1e-18:
        return ksi_old
    gd = float(g @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c
    ksi_trial = Dl / (tau * d2)
    ksi_trial = max(0.0, ksi_trial)
    if ksi_old <= ksi_trial:
        return ksi_old
    return min(ksi_trial, (1.0 - eps_ksi) * ksi_old)

def phi_mu(alpha, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma):
    gd = float(g @ d)
    d2 = float(d @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c
    term1 = (eta - 1.0) * alpha * beta * Dl
    term2 = (abs(1.0 - alpha) - 1.0 + alpha) * c1
    term3 = 0.5 * (tau * L + Gamma) * (alpha ** 2) * d2
    return term1 + term2 + term3

def cal_alpha_mu(d, eta, beta, ksi, tau, L, Gamma, theta_val, g, c, mu_eff, y):
    denom = (tau * L + Gamma)
    if denom <= 1e-12:
        return 0.0
    alpha_min = 2.0 * (1.0 - eta) * beta * ksi * tau / denom
    a = max(alpha_min, 0.0)
    while (
        phi_mu(1.1 * a, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma) < 0.0
        and (1.1 * a < alpha_min + theta_val * beta)
    ):
        a *= 1.1
    return float(a)

# ============================================================
# BLOCK SLICES / PRINTS
# ============================================================
def _block_slices():
    s = {}
    i = 0
    s["pde_phi"] = slice(i, i + K_CON_PHI); i += K_CON_PHI
    s["pde_Ts"]  = slice(i, i + K_CON_TS);  i += K_CON_TS
    s["pde_Tf"]  = slice(i, i + K_CON_TF);  i += K_CON_TF
    s["bc_phi"]  = slice(i, i + N_BC_PHI);  i += N_BC_PHI
    s["bc_Ts"]   = slice(i, i + N_BC_TS);   i += N_BC_TS
    s["bc_Tf"]   = slice(i, i + N_BC_TF);   i += N_BC_TF
    s["iface"]   = slice(i, i + N_IF);      i += N_IF
    s["ic"]      = slice(i, i + N_IC);      i += N_IC
    assert i == M_CON, (i, M_CON)
    return s

BLOCKS = _block_slices()

def _rms(x):
    return float(jnp.sqrt(jnp.mean(x * x) + DTYPE(1e-30)))

def _maxabs(x):
    return float(jnp.max(jnp.abs(x)))

def print_block_stats(prefix, c):
    def pr(name):
        sl = BLOCKS[name]
        v = c[sl]
        print(f"  {name:10s}: rms={_rms(v):.3e}  max={_maxabs(v):.3e}")
    print(prefix)
    pr("pde_phi")
    pr("pde_Ts")
    pr("pde_Tf")
    pr("bc_phi")
    pr("bc_Ts")
    pr("bc_Tf")
    pr("iface")
    pr("ic")

# ============================================================
# TRAIN SQP (FIXED BASE BATCHES, JITTER ONLY)
# ============================================================
def train_sqp(
    seed=0,
    hidden_dim=35,
    num_hidden=3,
    max_iters=700,
    print_every=10,
    beta_shift=200.0,
    beta_power=0.6,
    alpha_cap=1e1,
    L_lip=40.0,
    Gamma_lip=40.0,
    mu_damp=1e-4,
    jitter_every_iter=True,
    sx=1e-5,
    sy=1e-5,
    st=1e-5,
    sx_bc=1e-5,
    sy_bc=1e-5,
    st_bc=1e-5,
    sx_ic=1e-5,
    sy_ic=1e-5,
    sy_if=1e-5,
    st_if=1e-5,
    w_obj_phi=10.0,
    w_obj_Ts=10.0,
    w_obj_Tf=10.0,
    d_phi=0.05,
    sigma_a=1.0,
    rho_s_cp_s=1.0,
    rho_f_cp_f=1.0,
    k_s=0.50,
    k_f=0.15,
    gamma=10.0,
    vy=1.0,
):
    key = random.PRNGKey(seed)

    layer_sizes = [3] + [hidden_dim] * num_hidden + [3]
    key, k0 = random.split(key)
    params0 = init_mlp_params(k0, layer_sizes)
    theta, shapes = flatten_params(params0)

    n = theta.shape[0]
    H = jnp.eye(n, dtype=DTYPE)

    # ========================================================
    # FIXED base sets: sample ONCE only
    # ========================================================
    key, k_obj_phi, k_obj_Ts, k_obj_Tf = random.split(key, 4)
    X_obj_phi, _ = sample_stratified_3d(k_obj_phi, x_min, Ls, y_min, y_max, t_min, t_max, NX_OBJ_PHI, NY_OBJ_PHI, NT_OBJ_PHI)
    X_obj_Ts, _  = sample_stratified_3d(k_obj_Ts,  x_min, Ls, y_min, y_max, t_min, t_max, NX_OBJ_TS,  NY_OBJ_TS,  NT_OBJ_TS)
    X_obj_Tf, _  = sample_stratified_3d(k_obj_Tf,  Ls,    x_max, y_min, y_max, t_min, t_max, NX_OBJ_TF, NY_OBJ_TF, NT_OBJ_TF)

    key, k_con_phi, k_con_Ts, k_con_Tf = random.split(key, 4)
    X_con_phi, ids_con_phi = sample_stratified_3d(k_con_phi, x_min, Ls, y_min, y_max, t_min, t_max, NX_CON_PHI, NY_CON_PHI, NT_CON_PHI)
    X_con_Ts,  ids_con_Ts  = sample_stratified_3d(k_con_Ts,  x_min, Ls, y_min, y_max, t_min, t_max, NX_CON_TS,  NY_CON_TS,  NT_CON_TS)
    X_con_Tf,  ids_con_Tf  = sample_stratified_3d(k_con_Tf,  Ls,    x_max, y_min, y_max, t_min, t_max, NX_CON_TF, NY_CON_TF, NT_CON_TF)

    key, k1, k2, k3, k4, k5, k6, k7, k8, k9, k10, k11, k12, k13 = random.split(key, 14)

    X_phi_left, ids_phi_left = sample_bc_time_binned(k1, K_BC_PHI_LEFT, x_fixed=x_min, y_lo=y_min, y_hi=y_max)
    X_phi_y0,   ids_phi_y0   = sample_bc_time_binned(k2, K_BC_PHI_Y0, y_fixed=y_min, x_lo=x_min, x_hi=Ls)
    X_phi_y1,   ids_phi_y1   = sample_bc_time_binned(k3, K_BC_PHI_Y1, y_fixed=y_max, x_lo=x_min, x_hi=Ls)
    X_phi_if,   ids_phi_if   = sample_bc_time_binned(k4, K_BC_PHI_IF, x_fixed=Ls, y_lo=y_min, y_hi=y_max)

    X_Ts_x0, ids_Ts_x0 = sample_bc_time_binned(k5, K_BC_TS_X0, x_fixed=x_min, y_lo=y_min, y_hi=y_max)
    X_Ts_y0, ids_Ts_y0 = sample_bc_time_binned(k6, K_BC_TS_Y0, y_fixed=y_min, x_lo=x_min, x_hi=Ls)
    X_Ts_y1, ids_Ts_y1 = sample_bc_time_binned(k7, K_BC_TS_Y1, y_fixed=y_max, x_lo=x_min, x_hi=Ls)

    xL = float(DTYPE(Ls) + DTYPE(1e-8))
    xR = float(x_max - DTYPE(1e-8))
    X_Tf_y0, ids_Tf_y0 = sample_bc_time_binned(k8,  K_BC_TF_Y0, y_fixed=y_min, x_lo=xL, x_hi=xR)
    X_Tf_y1, ids_Tf_y1 = sample_bc_time_binned(k9,  K_BC_TF_Y1, y_fixed=y_max, x_lo=xL, x_hi=xR)
    X_Tf_x1, ids_Tf_x1 = sample_bc_time_binned(k10, K_BC_TF_X1, x_fixed=x_max, y_lo=y_min, y_hi=y_max)

    X_if, ids_if = sample_interface_yt(k11)

    X_ic_phi, ids_ic_phi = sample_ic_xy(k12, x_min, Ls, y_min, y_max, NX_IC, NY_IC)
    X_ic_Ts,  ids_ic_Ts  = sample_ic_xy(k13, x_min, Ls, y_min, y_max, NX_IC, NY_IC)
    key, k_ic_tf = random.split(key)
    X_ic_Tf,  ids_ic_Tf  = sample_ic_xy(k_ic_tf, Ls, x_max, y_min, y_max, NX_IC, NY_IC)

    eta, sigma = 0.25, 0.1
    eps_tau, eps_ksi = 1e-2, 1e-2
    theta_val = 10.0
    tau_k, ksi_k = 1.0, 1.0

    print("M_CON =", int(M_CON), "n_params =", int(n))
    print("Fixed-base mode: no resampling, fresh jitter only.")

    t0_clock = time.time()

    for it in range(1, max_iters + 1):
        k_beta = (it // 10) * 10
        beta_k = float(min(1.0, (beta_shift / (beta_shift + k_beta)) ** beta_power))

        if jitter_every_iter:
            key, *ks = random.split(key, 17)
            (
                kj_obj_phi, kj_obj_Ts, kj_obj_Tf,
                kj_con_phi, kj_con_Ts, kj_con_Tf,
                kj_phi_left, kj_phi_y0, kj_phi_y1, kj_phi_if,
                kj_Ts_x0, kj_Ts_y0, kj_Ts_y1,
                kj_Tf_y0, kj_Tf_y1, kj_Tf_x1
            ) = ks[:16]
            key, kj_if = random.split(key)
            key, kj_ic_phi = random.split(key)
            key, kj_ic_Ts = random.split(key)
            key, kj_ic_Tf = random.split(key)

            # Objective sets
            X_obj_phi_use = jitter_xyt(kj_obj_phi, X_obj_phi, sx, sy, st)
            X_obj_Ts_use  = jitter_xyt(kj_obj_Ts,  X_obj_Ts,  sx, sy, st)
            X_obj_Tf_use  = jitter_xyt(kj_obj_Tf,  X_obj_Tf,  sx, sy, st)

            # PDE anchors
            X_con_phi_use = jitter_xyt(kj_con_phi, X_con_phi, sx, sy, st)
            X_con_Ts_use  = jitter_xyt(kj_con_Ts,  X_con_Ts,  sx, sy, st)
            X_con_Tf_use  = jitter_xyt(kj_con_Tf,  X_con_Tf,  sx, sy, st)

            # Neutron BCs
            X_phi_left_use = project_x_fixed(jitter_xyt(kj_phi_left, X_phi_left, 0.0, sy_bc, st_bc), x_min)
            X_phi_y0_use   = project_y_fixed(jitter_xyt(kj_phi_y0, X_phi_y0, sx_bc, 0.0, st_bc), y_min, x_lo=x_min, x_hi=Ls)
            X_phi_y1_use   = project_y_fixed(jitter_xyt(kj_phi_y1, X_phi_y1, sx_bc, 0.0, st_bc), y_max, x_lo=x_min, x_hi=Ls)
            X_phi_if_use   = project_x_fixed(jitter_xyt(kj_phi_if, X_phi_if, 0.0, sy_bc, st_bc), Ls)

            # Fuel thermal BCs
            X_Ts_x0_use = project_x_fixed(jitter_xyt(kj_Ts_x0, X_Ts_x0, 0.0, sy_bc, st_bc), x_min)
            X_Ts_y0_use = project_y_fixed(jitter_xyt(kj_Ts_y0, X_Ts_y0, sx_bc, 0.0, st_bc), y_min, x_lo=x_min, x_hi=Ls)
            X_Ts_y1_use = project_y_fixed(jitter_xyt(kj_Ts_y1, X_Ts_y1, sx_bc, 0.0, st_bc), y_max, x_lo=x_min, x_hi=Ls)

            # Coolant thermal BCs
            X_Tf_y0_use = project_y_fixed(jitter_xyt(kj_Tf_y0, X_Tf_y0, sx_bc, 0.0, st_bc), y_min, x_lo=xL, x_hi=xR)
            X_Tf_y1_use = project_y_fixed(jitter_xyt(kj_Tf_y1, X_Tf_y1, sx_bc, 0.0, st_bc), y_max, x_lo=xL, x_hi=xR)
            X_Tf_x1_use = project_x_fixed(jitter_xyt(kj_Tf_x1, X_Tf_x1, 0.0, sy_bc, st_bc), x_max)

            # Interface + IC
            X_if_use = project_interface(jitter_xyt(kj_if, X_if, 0.0, sy_if, st_if))
            X_ic_phi_use = project_t0(jitter_xyt(kj_ic_phi, X_ic_phi, sx_ic, sy_ic, 0.0))
            X_ic_Ts_use  = project_t0(jitter_xyt(kj_ic_Ts,  X_ic_Ts,  sx_ic, sy_ic, 0.0))
            X_ic_Tf_use  = project_t0(jitter_xyt(kj_ic_Tf,  X_ic_Tf,  sx_ic, sy_ic, 0.0))
        else:
            X_obj_phi_use, X_obj_Ts_use, X_obj_Tf_use = X_obj_phi, X_obj_Ts, X_obj_Tf
            X_con_phi_use, X_con_Ts_use, X_con_Tf_use = X_con_phi, X_con_Ts, X_con_Tf
            X_phi_left_use, X_phi_y0_use, X_phi_y1_use, X_phi_if_use = X_phi_left, X_phi_y0, X_phi_y1, X_phi_if
            X_Ts_x0_use, X_Ts_y0_use, X_Ts_y1_use = X_Ts_x0, X_Ts_y0, X_Ts_y1
            X_Tf_y0_use, X_Tf_y1_use, X_Tf_x1_use = X_Tf_y0, X_Tf_y1, X_Tf_x1
            X_if_use = X_if
            X_ic_phi_use, X_ic_Ts_use, X_ic_Tf_use = X_ic_phi, X_ic_Ts, X_ic_Tf

        obj_val, g = F_and_g_obj(
            theta, shapes,
            X_obj_phi_use, X_obj_Ts_use, X_obj_Tf_use,
            DTYPE(w_obj_phi), DTYPE(w_obj_Ts), DTYPE(w_obj_Tf),
            DTYPE(d_phi), DTYPE(sigma_a),
            DTYPE(rho_s_cp_s), DTYPE(k_s), DTYPE(gamma),
            DTYPE(rho_f_cp_f), DTYPE(k_f), DTYPE(vy)
        )

        args = (
            X_con_phi_use, ids_con_phi,
            X_con_Ts_use, ids_con_Ts,
            X_con_Tf_use, ids_con_Tf,
            X_phi_left_use, ids_phi_left,
            X_phi_y0_use, ids_phi_y0,
            X_phi_y1_use, ids_phi_y1,
            X_phi_if_use, ids_phi_if,
            X_Ts_x0_use, ids_Ts_x0,
            X_Ts_y0_use, ids_Ts_y0,
            X_Ts_y1_use, ids_Ts_y1,
            X_Tf_y0_use, ids_Tf_y0,
            X_Tf_y1_use, ids_Tf_y1,
            X_Tf_x1_use, ids_Tf_x1,
            X_if_use, ids_if,
            X_ic_phi_use, ids_ic_phi,
            X_ic_Ts_use, ids_ic_Ts,
            X_ic_Tf_use, ids_ic_Tf,
            DTYPE(d_phi), DTYPE(sigma_a),
            DTYPE(rho_s_cp_s), DTYPE(k_s), DTYPE(gamma),
            DTYPE(rho_f_cp_f), DTYPE(k_f), DTYPE(vy)
        )

        c, J = C_and_J(theta, shapes, *args)

        # no rescaling; fixed batch + jitter only
        c_s, J_s = c, J

        d, y = kkt_solve_once(H, J_s, g, c_s, DTYPE(mu_damp))

        dn = float(jnp.linalg.norm(d, jnp.inf))
        if dn > 1e-12:
            tau_k = cal_tau_mu(H, d, sigma, tau_k, eps_tau, g, c_s, DTYPE(mu_damp), y)
            ksi_k = cal_ksi_mu(d, tau_k, ksi_k, eps_ksi, g, c_s, DTYPE(mu_damp), y)
            alpha = cal_alpha_mu(d, eta, beta_k, ksi_k, tau_k, L_lip, Gamma_lip, theta_val, g, c_s, DTYPE(mu_damp), y)
        else:
            alpha = 0.0

        alpha = float(min(alpha, alpha_cap))
        theta = theta + DTYPE(alpha) * d

        if it % int(print_every) == 0:
            c_now = C_only(theta, shapes, *args)
            feas_raw = float(jnp.mean(c_now ** 2))
            station  = float(jnp.linalg.norm(g + J.T @ y, jnp.inf))

            print(f"[it={it}] obj={float(obj_val):.3e} feas_raw={feas_raw:.3e} alpha={alpha:.2e} beta={beta_k:.2e} station={station:.2e} dn={dn:.2e}")
            print_block_stats("blocks (raw c):", c_now)

            params_now = unflatten_params(theta, shapes)
            r_phi_obj = residual_phi(params_now, X_obj_phi_use, DTYPE(d_phi), DTYPE(sigma_a))
            r_Ts_obj  = residual_Ts(params_now, X_obj_Ts_use, DTYPE(rho_s_cp_s), DTYPE(k_s), DTYPE(gamma))
            r_Tf_obj  = residual_Tf(params_now, X_obj_Tf_use, DTYPE(rho_f_cp_f), DTYPE(k_f), DTYPE(vy))
            print(
                "Raw PDE RMS on obj:",
                "phi", _rms(r_phi_obj),
                "Ts", _rms(r_Ts_obj),
                "Tf", _rms(r_Tf_obj),
            )

    print(f"[done] elapsed={time.time() - t0_clock:.2f}s")
    return theta, shapes

# ============================================================
# MAIN
# ============================================================
def main():
    theta, shapes = train_sqp(
        seed=0,
        hidden_dim=35,
        num_hidden=3,
        max_iters=10000,
        print_every=10,
        beta_shift=400.0,
        beta_power=0.6,
        alpha_cap=1e1,
        L_lip=60.0,
        Gamma_lip=60.0,
        mu_damp=1e-4,
        jitter_every_iter=True,
        sx=5e-3,
        sy=5e-3,
        st=5e-3,
        sx_bc=5e-3,
        sy_bc=5e-3,
        st_bc=5e-3,
        sx_ic=5e-3,
        sy_ic=5e-3,
        sy_if=1e-5,
        st_if=1e-5,
        w_obj_phi=10.0,
        w_obj_Ts=10.0,
        w_obj_Tf=10.0,
        d_phi=0.05,
        sigma_a=1.0,
        rho_s_cp_s=1.0,
        rho_f_cp_f=1.0,
        k_s=0.50,
        k_f=0.15,
        gamma=10.0,
        vy=1.0,
    )

    np.save("theta_sqp_coupled_fixedbatch_jitter_deterministic.npy", np.array(theta))
    print("Saved theta -> theta_sqp_coupled_fixedbatch_jitter_deterministic.npy")

In [3]:
if __name__ == "__main__":
    main()

M_CON = 2048 n_params = 2768
Fixed-base mode: no resampling, fresh jitter only.


2026-05-29 02:46:08.106055: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 26.98GiB (rounded to 28970168064)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-05-29 02:46:08.106281: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] __________________________________________________________________*********________******xxx_______*
E0529 02:46:08.106312  240864 pjrt_stream_executor_client.cc:2916] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 28970168064 bytes. [tf-allocator-allocation-error='']


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 28970168064 bytes.

In [ ]:
2962.80s